# E1/E2 + Human–Judge GEE — Frozen Predictions Reanalysis (CPU)
**Purpose:** Reproduce the independent E1/E2 sensitivity and 60 exploratory human–judge GEE analyses, including **Prompt_ID cluster-bootstrap confidence intervals**, without calling any LLM and without modifying the final `results.rar` experiments. All outputs are archived in **one downloadable ZIP**.

**Runtime:** Google Colab → Runtime → Change runtime type → **CPU** (GPU not required). Internet required if using the public GitHub retrieval mode. The default is **repository download**, with an **upload mode** for an offline ZIP of `paired_cases_part_01.csv` through `paired_cases_part_11.csv`.

**Strict separation:** E1/E2 is **not** re-evaluated against FP16 `results.rar` reference predictions. Every intervention is compared with its own frozen E1/rot0 control. This notebook does not replace or touch data from any other experiment.

**Expected input:** 11 case-level CSV parts (10,408 rows, 20 paired contrasts) plus three audited analysis scripts retrieved from the repository. **Expected output:** `P2_E1E2_GEE_CPU_REANALYSIS_<timestamp>.zip` containing two primary result tables, one independent stability review, hashes, run metadata and execution logs.

In [ ]:
# 1. Set options — CPU-only; no model inference
SOURCE_MODE = 'github'  # 'github' or 'upload_zip'
REPOSITORY = 'fatmas1982/semantic-coherence-recovery'
REF = 'main'   # set a specific verified commit SHA for immutable reproducibility
E1E2_BOOTSTRAP_REPS = 3000
GEE_BOOTSTRAP_REPS = 2500
SEED_E1E2 = 20261009
SEED_GEE = 20261010
DOWNLOAD_AT_END = True

from pathlib import Path
ROOT = Path('/content/p2_cpu_reanalysis')
DATA = ROOT/'data/validity_2026_10'
SCRIPTS = ROOT/'code/validity'
OUTPUT = ROOT/'results/validity_2026_10'
for p in (DATA,SCRIPTS,OUTPUT): p.mkdir(parents=True,exist_ok=True)
print('Runtime: CPU statistical reanalysis; GPU not needed')

In [ ]:
# 2. Libraries
%pip -q install numpy pandas scipy scikit-learn statsmodels requests
import os,sys,json,hashlib,subprocess,zipfile,datetime,platform,importlib.metadata as im
import pandas as pd
import requests
from google.colab import files
assert E1E2_BOOTSTRAP_REPS>=100 and GEE_BOOTSTRAP_REPS>=100

In [ ]:
# 3. Acquire EXACT frozen data + code; FAIL if parts are missing
NEEDED_SCRIPTS=['reanalyse_e1e2_pairs_2026.py','reanalyse_hji_gee_2026.py','reanalyse_gee_stability_2026.py']
repo_url=f'https://raw.githubusercontent.com/{REPOSITORY}/{REF}'
session=requests.Session()
def retrieve(rel,dest):
    url=f'{repo_url}/{rel}'
    response=session.get(url,timeout=90)
    response.raise_for_status()
    if len(response.content)==0:raise RuntimeError('Empty file: '+url)
    dest.parent.mkdir(parents=True,exist_ok=True)
    dest.write_bytes(response.content)
    print('Retrieved',rel,len(response.content),'bytes')
for name in NEEDED_SCRIPTS:retrieve('code/validity/'+name,SCRIPTS/name)
if SOURCE_MODE=='github':
    for i in range(1,12):
        name=f'paired_cases_part_{i:02d}.csv'
        retrieve('data/validity_2026_10/'+name,DATA/name)
elif SOURCE_MODE=='upload_zip':
    print('Upload ONE ZIP containing the 11 paired_cases_part_XX.csv files')
    uploaded=files.upload()
    if len(uploaded)!=1 or not next(iter(uploaded)).lower().endswith('.zip'):
        raise ValueError('Expected exactly one ZIP upload')
    blob=next(iter(uploaded.values()))
    import io
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        for i in range(1,12):
            name=f'paired_cases_part_{i:02d}.csv'
            hits=[p for p in z.namelist() if Path(p).name==name]
            if len(hits)!=1:raise ValueError(f'Expected exactly one {name} in ZIP')
            (DATA/name).write_bytes(z.read(hits[0]))
else:raise ValueError('SOURCE_MODE must be github or upload_zip')

In [ ]:
# 4. Input quality gate: no duplicates, 20 contrasts, 60 expected axes
parts=sorted(DATA.glob('paired_cases_part_*.csv'))
if len(parts)!=11:raise AssertionError('11 parts required')
frames=[pd.read_csv(f,dtype={'case_id':str,'prompt_id':str}) for f in parts]
cases=pd.concat(frames,ignore_index=True)
keys=['judge','target','experiment','variant']
required=set(keys+['case_id','prompt_id','gold_label','pred_base','pred_pert','changed','primary_disagree','boundary_disagree','severity_disagree','generator_id','margin_base'])
missing=required-set(cases.columns)
assert not missing, f'Missing columns: {sorted(missing)}'
assert len(cases)==10408,(len(cases),'not 10408')
assert cases.groupby(keys).ngroups==20,'20 paired experiments expected'
assert not cases.duplicated(keys+['case_id']).any(),'Duplicate case within contrast'
assert ((cases.pred_base != cases.pred_pert).astype(int)==cases.changed.astype(int)).all(),'Change flag mismatch'
assert not cases[list(required)].isna().any().any(),'Missing values in required inputs'
print('PASS',len(parts),'input parts;',len(cases),'case-level comparison rows; 20 paired comparisons')
print(cases.groupby(['judge','target','experiment','variant']).size().to_string())

In [ ]:
# 5. Reproduce 20 E1/E2 paired comparisons; Prompt_ID-cluster bootstrap
import time
runs=[]
def run_step(label,argv):
    print('\n>>>',label)
    tic=time.time()
    proc=subprocess.run(argv,capture_output=True,text=True,cwd=ROOT)
    log=OUTPUT/(label+'.log')
    log.write_text('COMMAND: '+' '.join(argv)+'\n\nSTDOUT:\n'+proc.stdout+'\nSTDERR:\n'+proc.stderr+'\nRETURN CODE: '+str(proc.returncode),encoding='utf-8')
    print(proc.stdout[-3000:]); print(proc.stderr[-1200:] if proc.stderr else '')
    runs.append({'name':label,'returncode':proc.returncode,'seconds':round(time.time()-tic,2),'log':log.name})
    if proc.returncode:raise RuntimeError(f'{label} FAILED. Inspect {log}')
run_step('E1E2_20_RECOMPUTED',[
    sys.executable,str(SCRIPTS/'reanalyse_e1e2_pairs_2026.py'),
    '--data-dir',str(DATA),'--out',str(OUTPUT),
    '--bootstrap-reps',str(E1E2_BOOTSTRAP_REPS),'--seed',str(SEED_E1E2)])
assert len(pd.read_csv(OUTPUT/'E1E2_PAIRED_RECOMPUTED.csv'))==20

In [ ]:
# 6. Reproduce 60 exploratory GEE associations and class-adjusted bootstrap CI
run_step('HJI_GEE_60_RECOMPUTED',[
    sys.executable,str(SCRIPTS/'reanalyse_hji_gee_2026.py'),
    '--data-dir',str(DATA),'--out',str(OUTPUT),
    '--bootstrap-reps',str(GEE_BOOTSTRAP_REPS),'--seed',str(SEED_GEE)])
g=pd.read_csv(OUTPUT/'CLASS_ADJUSTED_HUMAN_JUDGE_60_CORRECTED_CI_20261010.csv')
assert len(g)==60
print('Holm significant:',int((g.gee_Holm_p_60<.05).sum()))
print('GEE convergence:',g.gee_converged.value_counts(dropna=False).to_dict())

In [ ]:
# 7. Independent stability refit: 60 fits + guarded exact comparison against repository reference
# Original stability script independently verifies GEE coefficients/OR/p against frozen reference.
# Its reference may be absent/renamed in certain repository revisions; any failure is logged, NOT silently ignored.
try:
    retrieve('results/validity_2026_10/HJI_GEE_STABILITY_60.csv',OUTPUT/'HJI_GEE_STABILITY_60.csv')
except Exception as exc:
    print('Reference table unavailable:',repr(exc),'— continue; script will determine if it requires this file')
try:
    run_step('HJI_GEE_STABILITY_60',[
        sys.executable,str(SCRIPTS/'reanalyse_gee_stability_2026.py'),
        '--data-dir',str(DATA),
        '--out',str(OUTPUT/'HJI_GEE_STABILITY_60_RECOMPUTED.csv')])
except Exception as exc:
    print('LIMITATION: independent GEE refit did not PASS:',exc)
    (OUTPUT/'GEE_STABILITY_LIMITATION.txt').write_text(str(exc))

In [ ]:
# 8. Audit summary and provenance — no manuscript modification, no RAR substitution
import numpy as np
paired=pd.read_csv(OUTPUT/'E1E2_PAIRED_RECOMPUTED.csv')
summary={
 'date_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'repository':REPOSITORY,'requested_ref':REF,'mode':SOURCE_MODE,
 'runtime':'CPU; no GPU, no LLM inference',
 'frozen_paired_case_rows':len(cases),'paired_contrasts':len(paired),'gee_tests':len(g),
 'gee_holm_significant_count':int((g.gee_Holm_p_60<.05).sum()),
 'e1e2_bootstrap_reps':E1E2_BOOTSTRAP_REPS,'gee_bootstrap_reps':GEE_BOOTSTRAP_REPS,
 'seed_e1e2':SEED_E1E2,'seed_gee':SEED_GEE,
 'scripts_and_data_sha256':{},'steps':runs,
 'limitations':['E1/E2 internal rot0 control used; FP16 RAR reference is NOT a paired baseline.',
   'GEE is exploratory, not causal; Holm correction covers 60 contrasts.',
   'Bootstrap intervals are conditional on the frozen case-level predictions.',
   'No entire-manuscript audit and no independent RAR equivalence performed.']
}
for p in sorted([*parts,*[SCRIPTS/x for x in NEEDED_SCRIPTS]]):
    summary['scripts_and_data_sha256'][str(p.relative_to(ROOT))]=hashlib.sha256(p.read_bytes()).hexdigest()
for lib in ['numpy','pandas','scipy','scikit-learn','statsmodels']:
    summary.setdefault('packages',{})[lib]=im.version(lib)
(OUTPUT/'RUN_MANIFEST.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print('20 paired comparisons:',paired[['judge','target','experiment','variant','changed_n','delta_macroF1']].to_string(index=False))
print('\nHolm-significant GEE findings:')
print(g.loc[g.gee_Holm_p_60<.05,['judge','target','experiment','variant','human_axis','gee_adjusted_OR','gee_Holm_p_60']].to_string(index=False))

In [ ]:
# 9. Package ALL outputs, scripts and input hashes into one ZIP; download automatically
import datetime
stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M%S_UTC')
archive=Path('/content')/f'P2_E1E2_GEE_CPU_REANALYSIS_{stamp}.zip'
required_results=[
    'E1E2_PAIRED_RECOMPUTED.csv',
    'CLASS_ADJUSTED_HUMAN_JUDGE_60_CORRECTED_CI_20261010.csv',
    'RUN_MANIFEST.json'
]
for name in required_results:
    assert (OUTPUT/name).is_file(),f'Missing required output: {name}'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=7) as z:
    for folder in [OUTPUT,SCRIPTS]:
        for f in sorted(folder.rglob('*')):
            if f.is_file():z.write(f,f.relative_to(ROOT))
    # Include input-manifest and SHA256 only, not duplicated raw case-level CSVs.
    readme="""CPU-only reanalysis of frozen E1/E2 case-level comparisons.
Input: 11 data/validity_2026_10/paired_cases_part_XX.csv (10,408 rows).
Outputs: E1E2 paired 20-row metrics with prompt-cluster bootstrap;
60 exploratory GEE fits with Holm adjustment and class-adjusted CI;
optional independent 60-fit GEE stability check; execution logs and SHA256 manifest.
The original input CSV files are not duplicated in this output ZIP; download
from the SHA256-identified repository inputs recorded in RUN_MANIFEST.json.
DO NOT combine E1/rot0 with final RAR FP16 reference runs for T1.
"""
    z.writestr('README_OUTPUTS.txt',readme)
print('OUTPUT:',archive,'bytes:',archive.stat().st_size)
print('Files:',zipfile.ZipFile(archive).namelist())
if DOWNLOAD_AT_END:files.download(str(archive))

## Interpretation / accepted limitations
- Successful execution means **frozen prediction reanalysis** completed, not that judge model inference was rerun.
- The 20 E1/E2 comparisons have internal paired baselines; the 60 GEE tests are **exploratory** and apply Holm correction. Small numbers of changed decisions can limit stable interpretation.
- No data from `results.rar` are required to rerun E1/E2 and GEE. An independent **RAR/CPU/GPU source-to-manuscript** audit is a separate task and should not be represented by these outputs.
- Download the ZIP after the last cell. If browser blocks automatic download, find the ZIP in `/content/` and download from the Colab Files sidebar.